In [2]:
import pandas as pd
import jenkspy
from pathlib import Path
import numpy as np

# ============================================================
# INPUT
# ============================================================
INPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/data/district_final_risk_score.csv")

df = pd.read_csv(INPUT_CSV)

# ============================================================
# FILTER SUMMER 2026
# ============================================================

summer = df[df["timeperiod"].isin([
    "2026_04",
    "2026_05",
    "2026_06",
    # "2026_07",
])].copy()

# ============================================================
# DISTRICT-WISE HEAT RISK
# (Change mean() to max() or sum() if desired)
# ============================================================

district_heat = (
    summer.groupby("district", as_index=False)["heat-days-score"]
    .mean()
)

# ============================================================
# NATURAL JENKS BREAKS
# # ============================================================

# N_CLASSES = 5

# breaks = jenkspy.jenks_breaks(
#     district_heat["heat-days-score"],
#     n_classes=N_CLASSES
# )

# district_heat["heat_risk_class"] = pd.cut(
#     district_heat["heat-days-score"],
#     bins=breaks,
#     labels=range(1, N_CLASSES + 1),
#     include_lowest=True
# )

# print(breaks)
# print(district_heat.head())

# ============================================================
# Z-SCORE BINNING
# ============================================================

mean = district_heat["heat-days-score"].mean()
std = district_heat["heat-days-score"].std()

district_heat["z_score"] = (
    district_heat["heat-days-score"] - mean
) / std

district_heat["heat_risk_class"] = pd.cut(
    district_heat["z_score"],
    bins=[-np.inf, -1, -0.5, 0.5, 1, np.inf],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

print(f"Mean = {mean:.4f}")
print(f"Std Dev = {std:.4f}")

print(district_heat.head())


# ============================================================
# SAVE
# ============================================================
OUTPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/archive/analysis_scripts/2026_summer_district_hazard_risk_classes.csv")
district_heat.to_csv(OUTPUT_CSV, index=False)

print("Classification completed")
# print(breaks)

print(f"\nSaved {len(district_heat)} districts to:")
print(OUTPUT_CSV.resolve())

Mean = 11.9540
Std Dev = 2.9056
     district  heat-days-score   z_score  heat_risk_class
0       BAKSA           7.9880 -1.364937                1
1     BARPETA           9.4885 -0.848519                2
2   BISWANATH          14.6170  0.916523                4
3  BONGAIGAON           9.6735 -0.784849                2
4      CACHAR          12.2055  0.086573                3
Classification completed

Saved 33 districts to:
/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/archive/analysis_scripts/~/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/archive/analysis_scripts/2026_summer_district_hazard_risk_classes.csv


In [ ]:
# monthwise heat days score to compare with imd atlas

In [4]:
import pandas as pd
from pathlib import Path

# ============================================================
# INPUT
# ============================================================

INPUT_CSV = Path(
    "~/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/data/district_final_risk_score.csv"
).expanduser()

df = pd.read_csv(INPUT_CSV)

# Extract month
df["month"] = df["timeperiod"].str[-2:]

# ============================================================
# APRIL
# ============================================================

april = (
    df[df["month"] == "04"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# MAY
# ============================================================

may = (
    df[df["month"] == "05"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# JUNE
# ============================================================

june = (
    df[df["month"] == "06"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# ALL MONTHS (JAN-DEC)
# ============================================================

all_months = (
    df[df["month"].isin([f"{i:02d}" for i in range(1, 13)])]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# COMBINE
# ============================================================

district_heat = pd.concat(
    [april, may, june, all_months],
    axis=1
).reset_index()

district_heat.columns = [
    "district",
    "April",
    "May",
    "June",
    "All_Months_Sum"
]

# ============================================================
# SAVE
# ============================================================

OUTPUT_CSV = Path(
    "~/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/archive/analysis_scripts/district_heat_days_summary.csv"
).expanduser()

district_heat.to_csv(OUTPUT_CSV, index=False)

print(district_heat.head())
print(f"\nSaved {len(district_heat)} districts to:")
print(OUTPUT_CSV.resolve())

     district    April      May    June  All_Months_Sum
0       BAKSA   86.302   78.918  62.142         860.166
1     BARPETA  111.060   93.740  74.300        1021.988
2   BISWANATH  109.181  109.953  95.768        1239.832
3  BONGAIGAON  121.457  101.340  79.056        1128.160
4      CACHAR  143.399  105.764  73.501        1215.370

Saved 33 districts to:
/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/RiskScoreModel/archive/analysis_scripts/district_heat_days_summary.csv
